# Partitura `<annot>` Failure and CAMAT Fix

This notebook demonstrates two things on the same MEI file:

1. Raw `partitura` parsing fails when `<annot>` tags are present.
2. `camat.partitura_backend.parse_files_partitura` succeeds by using the current preprocessing + retry pipeline (while still staying on the partitura backend).


In [1]:
from pathlib import Path
import re
import pandas as pd

# Local MEI with annotation tags (no network required)
MEI_PATH = Path('CAMAT_revamped/exports/score_with_annot_20251117_105451.mei')
print('File exists:', MEI_PATH.exists())
print('Path:', MEI_PATH)

txt = MEI_PATH.read_text(encoding='utf-8', errors='ignore')
print('Count <annot> tags:', len(re.findall(r'<annot\b', txt)))


File exists: True
Path: CAMAT_revamped\exports\score_with_annot_20251117_105451.mei
Count <annot> tags: 4


## 1) Raw partitura behavior

This should fail with `element ... annot is not yet supported`.


In [2]:
import partitura as pt

try:
    score = pt.load_score(str(MEI_PATH))
    print('UNEXPECTED: raw partitura parse succeeded. Parts:', len(score.parts))
except Exception as exc:
    print('Expected raw partitura failure:')
    print(type(exc).__name__, str(exc))


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Expected raw partitura failure:
Exception element {http://www.music-encoding.org/ns/mei}annot is not yet supported


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmei.py:255: UserWarning: The key signature is not encoded in None or in any ancestor scoreDef.
  warnings.warn(
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmei.py:258: UserWarning: A default key signature of C maj is set.
  warnings.warn("A default key signature of C maj is set.")


## 2) Current CAMAT partitura pipeline

Use strict partitura-only mode (`allow_music21_fallback=False`) to prove this is not using music21 fallback.


In [3]:
from camat.partitura_backend import parse_files_partitura

results, dfs_by_name, last_df = parse_files_partitura(
    [str(MEI_PATH)],
    backend='none',
    display_preview=False,
    show_progress=False,
    normalize_mensural_durations=True,
    inject_missing_meter_signature=True,
    default_meter_count=2,
    default_meter_unit=2,
    try_verovio_mei_conversion=True,
    verovio_mensural_to_cmn=True,
    allow_music21_fallback=False,
)

print('Parsed entries:', len(results))
print('DataFrames keys:', list(dfs_by_name.keys()))
print('last_df is None:', last_df is None)
if last_df is not None:
    print('Rows:', len(last_df), 'Columns:', list(last_df.columns))
    display(last_df.head(10))


Processing (partitura): score_with_annot_20251117_105451.mei -> 00_score_with_annot_20251117_105451
Injected default meter signature into 4 tag(s): 2/2.
Verovio MEI postprocess: removed 4 <annot> element(s).
Parsed entries: 1
DataFrames keys: ['00_score_with_annot_20251117_105451']
last_df is None: False
Rows: 235 Columns: ['Measure', 'Local Onset', 'Global Onset', 'Duration', 'Pitch', 'MIDI', 'Voice', 'xml_id']


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmei.py:255: UserWarning: The key signature is not encoded in None or in any ancestor scoreDef.
  warnings.warn(
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmei.py:258: UserWarning: A default key signature of C maj is set.
  warnings.warn("A default key signature of C maj is set.")


,Measure,Local Onset,Global Onset,Duration,Pitch,MIDI,Voice,xml_id
0,1,3.0,-1.0,0.5,D4,62,P2 - Voice 2,d1e93
1,1,3.0,-1.0,1.0,F4,65,P2 - Voice 1,d1e92
2,1,3.0,-1.0,1.0,A4,69,P1 - Voice 2,d1e91
3,1,3.0,-1.0,1.0,D5,74,P1 - Voice 1,d1e64
4,1,3.5,-0.5,0.5,C4,60,P2 - Voice 2,d1e94
5,1,0.0,0.0,1.0,B3,59,P2 - Voice 2,d1e59
6,1,0.0,0.0,1.0,D4,62,P1 - Voice 2,d1e487
7,1,0.0,0.0,1.0,F4,65,P2 - Voice 1,d1e54
8,1,0.0,0.0,1.0,D5,74,P1 - Voice 1,d1e366
9,1,1.0,1.0,0.5,D4,62,P1 - Voice 2,d1e501


## 3) Optional: inspect conversion/postprocess stats

This cell uses internal helpers only for debug/demo purposes.


In [4]:
from camat.partitura_backend import _convert_mei_with_verovio_for_partitura

converted_path, cleanup_fn, removed_annots, wrapped_staff_groups = _convert_mei_with_verovio_for_partitura(
    str(MEI_PATH),
    mensural_to_cmn=True,
)

print('Converted temp path:', converted_path)
print('Removed <annot> count:', removed_annots)
print('Wrapped section-level staff groups:', wrapped_staff_groups)

tmp_txt = Path(converted_path).read_text(encoding='utf-8', errors='ignore')
print('Remaining <annot> tags in converted temp MEI:', len(re.findall(r'<annot\b', tmp_txt)))

if cleanup_fn:
    cleanup_fn()


Converted temp path: C:\Users\egorp\AppData\Local\Temp\tmptnjg3wwn.mei
Removed <annot> count: 4
Wrapped section-level staff groups: 0
Remaining <annot> tags in converted temp MEI: 0
